# M1: Ingesta TCGA-BRCA - GEN-MAMA-MX

Este notebook automatiza la descarga y procesamiento de datos de mutaciones somáticas del proyecto **TCGA-BRCA** (Breast Invasive Carcinoma) utilizando la API del **GDC (Genomic Data Commons)**.

**Objetivos:**
1. Descargar datos de mutaciones somáticas filtrados para variantes missense.
2. Identificar los genes más frecuentemente mutados.
3. Priorizar las variantes para el análisis estructural (Módulo M3).

In [ ]:
# @title 💾 Configuración de Persistencia y Entorno
# @markdown Marca esta casilla para guardar resultados en Google Drive. De lo contrario se usarán rutas locales temporales.
USE_DRIVE = True # @param {type:"boolean"}

import os
IS_COLAB = 'google.colab' in str(get_ipython())
BASE_PATH = "/content"

if IS_COLAB:
    if USE_DRIVE:
        from google.colab import drive
        print("Montando Google Drive...")
        drive.mount('/content/drive')
        BASE_PATH = "/content/drive/MyDrive/GEMAMX"
        print(f"Usando ruta persistente: {BASE_PATH}")
    
    # Limpiar y Crear carpetas usando rutas absolutas
    # Esto evita el error de "Too many levels of symbolic links"
    SCRIPTS_DIR = os.path.join(BASE_PATH, "scripts")
    DATA_DIR = os.path.join(BASE_PATH, "data")
    
    !mkdir -p {SCRIPTS_DIR} {DATA_DIR}/fasta {DATA_DIR}/structures
    
    # Crear enlaces simbólicos SOLO si no existen y no son circulares
    # Esto permite que los comandos relativos como '!python scripts/...' sigan funcionando
    for folder in ["scripts", "data"]:
        target = os.path.join(BASE_PATH, folder)
        link = os.path.join("/content", folder)
        if USE_DRIVE:
            # Eliminar lo que haya en la ruta local para poner el link
            if os.path.exists(link) or os.path.islink(link):
                !rm -rf {link}
            # Crear link simbólico de /content/scripts -> /content/drive/MyDrive/GEMAMX/scripts
            if not os.path.exists(link):
                os.symlink(target, link)
                print(f"Enlace creado: {link} -> {target}")
                
    print("✅ Entorno configurado correctamente.")

## 0. Configuración del Entorno (Colab)

Si estás ejecutando este notebook en Google Colab, la siguiente celda configurará automáticamente las carpetas necesarias y recreará los scripts de soporte.

In [ ]:
import os
import subprocess

IS_COLAB = 'google.colab' in str(get_ipython())

if IS_COLAB:
    print("Entorno Google Colab detectado. Configurando carpetas...")
    
    # Escribir el script de descarga de FASTA
    with open("scripts/fetch_fasta_colab.py", "w") as f:
        f.write(fasta_script)
    
    # Escribir el script de predicción automáticamente
    with open("scripts/predict_structures.py", "w") as f:
        f.write(predict_script)
    
    # Instalar dependencias necesarias
    print("Instalando dependencias (biopython)...")
    subprocess.run(["pip", "install", "biopython"], check=True)
    

    # Escribir el script de Docking


In [2]:
import pandas as pd
import requests
import json
import os
from tqdm import tqdm

print("Librerías cargadas correctamente.")

Librerías cargadas correctamente.


## 1. Configuración de la API del GDC

Definimos los parámetros para consultar el portal GDC y obtener las mutaciones somáticas (SSM) del proyecto BRCA.

In [3]:
def fetch_ssms(project_id="TCGA-BRCA", size=1000):
    """Consulta la API de GDC para obtener Simple Somatic Mutations."""
    ssm_url = "https://api.gdc.cancer.gov/ssms"
    
    # Filtros: Proyecto TCGA-BRCA y Variantes Missense
    filters = {
        "op": "and",
        "content": [
            {
                "op": "in",
                "content": {
                    "field": "occurrence.case.project.project_id",
                    "value": [project_id]
                }
            },
            {
                "op": "in",
                "content": {
                    "field": "consequence.transcript.consequence_type",
                    "value": ["missense_variant"]
                }
            }
        ]
    }
    
    params = {
        "filters": json.dumps(filters),
        "fields": "genomic_dna_change,consequence.transcript.gene.symbol,consequence.transcript.aa_change,occurrence.case.project.project_id",
        "format": "JSON",
        "size": size
    }
    
    response = requests.get(ssm_url, params=params)
    return response.json()

print("Iniciando descarga de datos de GDC...")
data = fetch_ssms(size=2000)
print(f"Descarga completada. Registros obtenidos: {len(data['data']['hits'])}")

Iniciando descarga de datos de GDC...
Descarga completada. Registros obtenidos: 2000


## 2. Procesamiento y Limpieza de Datos

Transformamos el JSON en un DataFrame de Pandas para su manipulación.

In [4]:
hits = data['data']['hits']
rows = []

for hit in hits:
    gene_symbol = hit['consequence'][0]['transcript']['gene']['symbol']
    aa_change = hit['consequence'][0]['transcript'].get('aa_change', 'N/A')
    dna_change = hit['genomic_dna_change']
    
    rows.append({
        "Gene": gene_symbol,
        "AA_Change": aa_change,
        "DNA_Change": dna_change
    })

df = pd.DataFrame(rows)
df.dropna(inplace=True)

# Guardar dataset completo
df.to_csv("mutations_brca.csv", index=False)
df.head(10)

,Gene,AA_Change,DNA_Change
0,SEPTIN14,I12M,chr7:g.55861961T>C
1,CLK1,N/A,chr2:g.200861318A>C
2,KCNJ16,E315Q,chr17:g.70133030G>C
3,CX3CR1,N/A,chr3:g.39281685G>A
4,CNKSR2,N/A,chrX:g.21432763C>T
5,ADH7,N/A,chr4:g.99420574C>T
6,APC,A1366V,chr5:g.112839691C>T
7,SEMA4B,N203H,chr15:g.90228002A>C
8,SYT16,N/A,chr14:g.62081147G>A
9,CSMD1,F458V,chr8:g.3493702A>C


## 3. Priorización de Variantes (M2)

Calculamos la frecuencia de las mutaciones para identificar los hotspots más relevantes del proyecto.

In [5]:
# Filtrar variantes que no tienen cambio de AA (N/A) antes de priorizar
df_filtered = df[df['AA_Change'] != 'N/A'].copy()

top_variants = df_filtered.groupby(['Gene', 'AA_Change']).size().reset_index(name='Frequency')
top_variants = top_variants.sort_values(by='Frequency', ascending=False).head(10)

print("Top 10 Variantes Priorizadas (con cambio proteico):")
print(top_variants)

# Guardar resultados para el siguiente módulo
top_variants_dict = top_variants.to_dict(orient='records')
with open("top_variants.json", "w") as f:
    json.dump(top_variants_dict, f, indent=4)

print("\nArchivo 'top_variants.json' generado con variantes válidas.")

Top 10 Variantes Missense en TCGA-BRCA:
         Gene AA_Change  Frequency
132      ARSK       N/A          3
648    GNPTAB       N/A          2
18     ACADSB       N/A          2
607     FOXA1       N/A          2
1499  SHROOM3       N/A          2
958      MEI1       N/A          2
1622    SUGP2       N/A          2
1208   PEX11B       N/A          2
536     ERCC6       N/A          2
602      FMN2       N/A          2

Archivos 'mutations_brca.csv' y 'top_variants.json' generados.


## 3.5 Generación de Secuencias FASTA

Este paso descarga las secuencias de UniProt para los genes priorizados y aplica las mutaciones correspondientes para generar los archivos `.fasta` necesarios para Boltz-1.

In [ ]:
# Ejecutar descarga de FASTA
!python scripts/fetch_fasta_colab.py

## 4. Predicción de Estructuras (Boltz-1)

Este bloque ejecuta la predicción 3D de forma automática. 

**Nota:** Si ejecutaste la celda de **Configuración del Entorno** al inicio, el script ya debería estar en su lugar.

In [ ]:
import os
# Intentar localizar el script en el entorno de Colab o Local
script_path = "scripts/predict_structures.py"

if os.path.exists(script_path):
    print(f"Ejecutando script desde: {script_path}")
    !python {script_path}
else:
    print("ERROR: No se encontró 'scripts/predict_structures.py'.")
    print("Por favor, revisa que la carpeta 'scripts' exista o ejecuta la sección de Configuración al inicio del notebook.")

## 5. Módulo M4: Docking Molecular (V3 - RECURSIVO)

Utilizamos **AutoDock Vina** para simular la interacción entre las proteínas predichas (WT y MUT) y ligandos de interés (ej. fármacos inhibidores).

In [ ]:
import os, glob
print('--- INICIANDO DOCKING V3 (RECURSIVO) ---')
check_path = 'data/structures'
pdbs = glob.glob(f'{check_path}/**/*.pdb', recursive=True)
print(f'Buscando en: {os.path.abspath(check_path)}')
print(f'Archivos encontrados: {len(pdbs)}')

if len(pdbs) == 0:
    print('!!! ERROR: No se encontraron PDBs. Mostrando contenido de data/:')
    !ls -R data/
else:
    !python scripts/run_docking_colab.py

## 6. Módulo M5: Reporte Final y Análisis de Binding

Consolidamos los resultados del RMSD estructural y las afiniades de docking para priorizar las variantes con mayor impacto funcional y farmacológico.

In [ ]:
import glob
import pandas as pd
import os

results = []
DOCKING_PATH = "data/docking"
logs = glob.glob(f"{DOCKING_PATH}/*_log.txt")

print(f"Escaneando logs en: {os.path.abspath(DOCKING_PATH)}")
print(f"Logs encontrados: {len(logs)}")

if len(logs) == 0:
    print("\n!!! ERROR: No se encontraron resultados de Docking.")
    print("Causas posibles:")
    print("1. No se han generado los PDBs en M3 (Boltz-1).")
    print("2. El script de M4 no encontró archivos PDB.")
    print("3. La carpeta 'data/docking' está vacía.")
else:
    for log in logs:
        name = os.path.basename(log).replace("_log.txt", "")
        with open(log, 'r') as f:
            content = f.read()
            if "Affinity:" in content:
                affinity = content.split("Affinity:")[-1].split("kcal/mol")[0].strip()
                results.append({"Variant": name, "Binding_Affinity (kcal/mol)": affinity})
    
    if results:
        df_results = pd.DataFrame(results)
        print("\nResultados Consolidados de Docking:")
        display(df_results)
        
        df_results.to_json("final_pipeline_results.json", orient="records")
        print(f"\nSUCCESS: Reporte 'final_pipeline_results.json' generado ({len(results)} registros).")
    else:
        print("\nLogs encontrados pero con formato incorrecto.")